# Nettoyage des Données 

Nettoyage et préparation du jeu de données Google Maps Reviews pour l'ingénierie des caractéristiques.

## Importations et Configuration

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Toutes les importations réussies !")

✅ Toutes les importations réussies !


## Charger données brutes

In [14]:
# Charger les données brutes
reviews = pd.read_csv('../data/raw/reviews.csv')
accounts = pd.read_csv('../data/raw/accounts.csv')

print(f" DIMENSIONS DES DONNEES ORIGINALES :")
print(f"  Reviews : {reviews.shape}")
print(f"  Accounts : {accounts.shape}")

# Afficher un échantillon
print("\n ECHANTILLON D'AVIS :")
display(reviews.head(3))

print("\n ECHANTILLON DE COMPTES :")
display(accounts.head(3))

 DIMENSIONS DES DONNEES ORIGINALES :
  Reviews : (17979, 17)
  Accounts : (605, 7)

 ECHANTILLON D'AVIS :


,_id,account_id,approximate_localization.lat,approximate_localization.lon,censored_text,cluster,content,content_not_full,content_translated,date,is_real,localization_missing,not_in_poland,photos_urls,rating,response_content,type_of_object
0,63e2e0d98b0803f03c00c02f,63e546a08f7e73edc5913876,53.3,15.9,False,Shops,Ok,False,False,2023-01-10T20:17:09.732Z,True,False,False,[],4,NaN,Dyskont spożywczy
1,63f2bcae8047cae56c427a1d,63e546a08f7e73edc5913876,53.6,15.6,False,Other services,Ok,False,False,2023-01-10T20:17:09.513Z,True,False,False,[],4,NaN,Magazyn
2,63f2bdc8ec2561ca0a25eb27,63e546a08f7e73edc5913876,53.6,15.7,False,Leisure,NaN,False,False,2022-11-11T20:17:09.942Z,True,False,False,[],5,NaN,Stadnina koni



 ECHANTILLON DE COMPTES :


,_id,is_deleted,is_private,is_real,local_guide_level,name_score,number_of_reviews
0,63e546a08f7e73edc5913876,False,False,True,5.0,744861,78.0
1,63e546e28f7e73edc59138c4,False,False,True,0.0,0,6.0
2,63e546f98f7e73edc59138cb,False,False,True,8.0,813520,532.0


## Etape 1: Analyse des doublons

In [15]:
print("=" * 80)
print("ETAPE 1 : SUPPRESSION DES DOUBLONS")
print("=" * 80)

# Vérifier les doublons dans les avis (en utilisant _id comme identifiant)
dups_reviews = reviews['_id'].duplicated().sum()
print(f"\n AVIS :")
print(f"  Duplicates : {dups_reviews}")

# Vérifier les doublons dans les comptes
dups_accounts = accounts['_id'].duplicated().sum()
print(f"\n  COMPTES :")
print(f"  Duplicates : {dups_accounts}")

# Supprimer les doublons
reviews_clean = reviews.drop_duplicates(subset=['_id'])
accounts_clean = accounts.drop_duplicates(subset=['_id'])

print(f"\n✅ APRES SUPPRESSION DES DOUBLONS :")
print(f"  Reviews : {len(reviews)} → {len(reviews_clean)} (-{len(reviews) - len(reviews_clean)})")
print(f"  Accounts : {len(accounts)} → {len(accounts_clean)} (-{len(accounts) - len(accounts_clean)})")

ETAPE 1 : SUPPRESSION DES DOUBLONS

 AVIS :
  Duplicates : 0

  COMPTES :
  Duplicates : 0

✅ APRES SUPPRESSION DES DOUBLONS :
  Reviews : 17979 → 17979 (-0)
  Accounts : 605 → 605 (-0)


## Etape  2 : Analyse des valeurs manquantes

In [16]:
print("\n" + "=" * 80)
print("ETAPE 2 : GESTION DES VALEURS MANQUANTES")
print("=" * 80)

# Vérifier les valeurs manquantes dans les avis
print(f"\n VALEURS MANQUANTES DES AVIS :")
missing_reviews = reviews_clean.isnull().sum()
if missing_reviews.sum() == 0:
    print("  ✅ Aucune valeur manquante !")
else:
    for col, count in missing_reviews[missing_reviews > 0].items():
        pct = (count / len(reviews_clean)) * 100
        print(f"  • {col}: {count:,} ({pct:.2f}%)")

# Vérifier les valeurs manquantes dans les comptes
print(f"\n VALEURS MANQUANTES DES COMPTES :")
missing_accounts = accounts_clean.isnull().sum()
if missing_accounts.sum() == 0:
    print("  ✅ Aucune valeur manquante !")
else:
    for col, count in missing_accounts[missing_accounts > 0].items():
        pct = (count / len(accounts_clean)) * 100
        print(f"  • {col}: {count:,} ({pct:.2f}%)")


ETAPE 2 : GESTION DES VALEURS MANQUANTES

 VALEURS MANQUANTES DES AVIS :
  • approximate_localization.lat: 117 (0.65%)
  • approximate_localization.lon: 117 (0.65%)
  • content: 4,628 (25.74%)
  • photos_urls: 2,451 (13.63%)
  • response_content: 17,323 (96.35%)

 VALEURS MANQUANTES DES COMPTES :
  • local_guide_level: 132 (21.82%)
  • number_of_reviews: 161 (26.61%)


## Etape 3 : Nettoyage du texte

In [ ]:
print("\n" + "=" * 80)
print("ETAPE 3 : NETTOYAGE DU TEXTE")
print("=" * 80)

# Renommer 'content' en 'review_text' pour plus de clarté
reviews_clean = reviews_clean.rename(columns={'content': 'review_text', '_id': 'review_id'})

# Supprimer les lignes avec des valeurs critiques manquantes
initial_len = len(reviews_clean)
reviews_clean = reviews_clean.dropna(subset=['review_text', 'rating', 'is_real'])
removed_critical = initial_len - len(reviews_clean)

print(f"\n Nettoyage du texte :")
print(f"  Lignes supprimées (données critiques manquantes) : {removed_critical:,}")

# Nettoyer le texte des avis
reviews_clean['review_text'] = reviews_clean['review_text'].astype(str).str.strip()
reviews_clean['review_text'] = reviews_clean['review_text'].str.replace(r'\s+', ' ', regex=True)

# Supprimer les avis vides
initial_len = len(reviews_clean)
reviews_clean = reviews_clean[reviews_clean['review_text'].str.len() > 0]
reviews_clean = reviews_clean[reviews_clean['review_text'] != 'nan']
removed_empty = initial_len - len(reviews_clean)

print(f"  Avis vides supprimés : {removed_empty:,}")
print(f"\n✅ Texte nettoyé !")

# Échantillon de texte nettoyé
print(f"\n ÉCHANTILLON DE TEXTE NETTOYÉ :")
for i, text in enumerate(reviews_clean['review_text'].head(3).values, 1):
    display_text = text[:100] + '...' if len(text) > 100 else text
    print(f"  {i}. {display_text}")


ÉTAPE 3 : NETTOYAGE DU TEXTE

 Nettoyage du texte :
  Lignes supprimées (données critiques manquantes) : 4,628
  Avis vides supprimés : 0

✅ Texte nettoyé !

 ÉCHANTILLON DE TEXTE NETTOYÉ :
  1. Ok
  2. Ok
  3. Ok


## Etape 4 : Validation des notes

In [18]:
print("\n" + "=" * 80)
print("ETAPE 4 : VALIDATION DES NOTES")
print("=" * 80)

# Vérifier la plage des notes
print(f"\n ANALYSE DES NOTES :")
print(f"  Unique ratings : {sorted(reviews_clean['rating'].unique())}")
print(f"  Min : {reviews_clean['rating'].min()}")
print(f"  Max : {reviews_clean['rating'].max()}")

# Supprimer les notes invalides (doit être 1-5)
initial_len = len(reviews_clean)
reviews_clean = reviews_clean[(reviews_clean['rating'] >= 1) & (reviews_clean['rating'] <= 5)]
removed_invalid = initial_len - len(reviews_clean)

print(f" Removed invalid ratings : {removed_invalid:,}")

# Distribution des notes
print(f"\n DISTRIBUTION DES NOTES :")
print(reviews_clean['rating'].value_counts().sort_index())


ETAPE 4 : VALIDATION DES NOTES

 ANALYSE DES NOTES :
  Unique ratings : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Min : 1
  Max : 5
 Removed invalid ratings : 0

 DISTRIBUTION DES NOTES :
rating
1     328
2     251
3     711
4    2143
5    9918
Name: count, dtype: int64


## Etape 5 : Validation des étiquettes

In [19]:
print("\n" + "=" * 80)
print("ETAPE 5 : VALIDATION DES LABELS")
print("=" * 80)

# S'assurer que is_real est binaire (convertir True/False en 1/0)
reviews_clean['is_real'] = reviews_clean['is_real'].astype(int)
accounts_clean['is_real'] = accounts_clean['is_real'].astype(int)

print(f"\n✅ REVIEWS is_real :")
print(reviews_clean['is_real'].value_counts())

print(f"\n✅ ACCOUNTS is_real :")
print(accounts_clean['is_real'].value_counts())


ETAPE 5 : VALIDATION DES LABELS

✅ REVIEWS is_real :
is_real
1    11101
0     2250
Name: count, dtype: int64

✅ ACCOUNTS is_real :
is_real
0    319
1    286
Name: count, dtype: int64


## Etape 6 : Nettoyage des comptes

In [ ]:
print("\n" + "=" * 80)
print("ETAPE 6 : NETTOYAGE DES COMPTES")
print("=" * 80)

# Renommer _id en account_id_orig pour les comptes
accounts_clean = accounts_clean.rename(columns={'_id': 'account_id_orig'})

# Supprimer les données critiques manquantes
initial_len = len(accounts_clean)
accounts_clean = accounts_clean.dropna(subset=['account_id_orig', 'is_real'])
removed = initial_len - len(accounts_clean)

print(f"\n  Removed rows with missing critical data : {removed:,}")

# Remplir les colonnes numériques manquantes avec la médiane
print(f"\n   Filling missing numeric columns...")
numeric_cols = accounts_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    missing = accounts_clean[col].isnull().sum()
    if missing > 0:
        accounts_clean[col] = accounts_clean[col].fillna(accounts_clean[col].median())
        print(f"    • {col} : {missing} valeurs remplies")

print(f"\n✅ Accounts cleaned!")
print(f"  Final shape: {accounts_clean.shape}")


ÉTAPE 6 : NETTOYAGE DES COMPTES

  Removed rows with missing critical data : 0

   Filling missing numeric columns...
    • local_guide_level : 132 valeurs remplies
    • number_of_reviews : 161 valeurs remplies

✅ Accounts cleaned!
  Final shape: (605, 7)


## Etape 7 : Fusion des données

In [21]:
print("\n" + "=" * 80)
print("ETAPE 7 : FUSION DES DONNÉES")
print("=" * 80)

print(f"\nBefore merge :")
print(f"  Reviews : {len(reviews_clean):,}")
print(f"  Accounts : {len(accounts_clean):,}")

# Fusionner sur account_id
merged = reviews_clean.merge(
    accounts_clean,
    left_on='account_id',
    right_on='account_id_orig',
    how='left',
    suffixes=('', '_account')
)

print(f"\nAfter merge :")
print(f"  Merged dataset : {len(merged):,}")
print(f"  Columns : {len(merged.columns)}")

print(f"\n✅ Datasets merged successfully!")


ETAPE 7 : FUSION DES DONNÉES

Before merge :
  Reviews : 13,351
  Accounts : 605

After merge :
  Merged dataset : 13,351
  Columns : 24

✅ Datasets merged successfully!


## Etape 8 : Gestion des valeurs manquantes restantes

In [22]:
print("\n" + "=" * 80)
print("ETAPE 8 : GESTION DES VALEURS MANQUANTES RESTANTES")
print("=" * 80)

# Vérifier les valeurs manquantes restantes
missing_total = merged.isnull().sum()
missing_cols = missing_total[missing_total > 0]

if len(missing_cols) == 0:
    print("\n✅ Aucune valeur manquante restante !")
else:
    print(f"\n⚠️  Valeurs manquantes restantes :")
    for col, count in missing_cols.items():
        pct = (count / len(merged)) * 100
        print(f"  • {col} : {count:,} ({pct:.2f}%)")
    
    # Remplir les valeurs manquantes
    print(f"\n  Remplissage des valeurs manquantes...")
    for col in missing_cols.index:
        if merged[col].dtype in ['int64', 'float64']:
            median_val = merged[col].median()
            merged[col] = merged[col].fillna(median_val)
            print(f"    • {col} : rempli avec la médiane ({median_val:.2f})")
        else:
            merged[col] = merged[col].fillna('INCONNU')
            print(f"    • {col} : rempli avec 'INCONNU'")
    
    print(f"\n✅ Toutes les valeurs manquantes traitées !")


ETAPE 8 : GESTION DES VALEURS MANQUANTES RESTANTES

⚠️  Valeurs manquantes restantes :
  • approximate_localization.lat : 84 (0.63%)
  • approximate_localization.lon : 84 (0.63%)
  • photos_urls : 2,262 (16.94%)
  • response_content : 12,763 (95.60%)

  Remplissage des valeurs manquantes...
    • approximate_localization.lat : rempli avec la médiane (51.90)
    • approximate_localization.lon : rempli avec la médiane (18.80)
    • photos_urls : rempli avec 'INCONNU'
    • response_content : rempli avec 'INCONNU'

✅ Toutes les valeurs manquantes traitées !


## Statistiques finales

In [23]:
print("\n" + "=" * 80)
print("STATISTIQUES FINALES DES DONNÉES")
print("=" * 80)

# Distribution des étiquettes
print(f"\n DISTRIBUTION DES LABELS (is_real) :")
label_counts = merged['is_real'].value_counts()
label_pct = merged['is_real'].value_counts(normalize=True) * 100

print(f"  Real  (1) : {label_counts[1]:,} ({label_pct[1]:.2f}%)")
print(f"  Fake  (0) : {label_counts[0]:,} ({label_pct[0]:.2f}%)")
print(f"  Imbalance : {label_counts[1]/label_counts[0]:.2f}:1")

# Informations sur les données
print(f"\n INFORMATIONS DU JEU DE DONNÉES :")
print(f"  Rows : {len(merged):,}")
print(f"  Columns : {len(merged.columns)}")
print(f"  Memory : {merged.memory_usage(deep=True).sum() / 1024**2:.2f} Mo")

# Valeurs manquantes
print(f"\n✅ VALEURS MANQUANTES : {merged.isnull().sum().sum()} au total")

# Afficher les données finales
print(f"\n APERÇU DU JEU DE DONNÉES FINAL :")
display(merged.head())


STATISTIQUES FINALES DES DONNÉES

 DISTRIBUTION DES LABELS (is_real) :
  Real  (1) : 11,101 (83.15%)
  Fake  (0) : 2,250 (16.85%)
  Imbalance : 4.93:1

 INFORMATIONS DU JEU DE DONNÉES :
  Rows : 13,351
  Columns : 24
  Memory : 11.13 Mo

✅ VALEURS MANQUANTES : 0 au total

 APERÇU DU JEU DE DONNÉES FINAL :


,review_id,account_id,approximate_localization.lat,approximate_localization.lon,censored_text,cluster,review_text,content_not_full,content_translated,date,...,rating,response_content,type_of_object,account_id_orig,is_deleted,is_private,is_real_account,local_guide_level,name_score,number_of_reviews
0,63e2e0d98b0803f03c00c02f,63e546a08f7e73edc5913876,53.3,15.9,False,Shops,Ok,False,False,2023-01-10T20:17:09.732Z,...,4,INCONNU,Dyskont spożywczy,63e546a08f7e73edc5913876,False,False,1,5.0,744861,78.0
1,63f2bcae8047cae56c427a1d,63e546a08f7e73edc5913876,53.6,15.6,False,Other services,Ok,False,False,2023-01-10T20:17:09.513Z,...,4,INCONNU,Magazyn,63e546a08f7e73edc5913876,False,False,1,5.0,744861,78.0
2,63f2bdf3ec2561ca0a25eb3d,63e546a08f7e73edc5913876,54.1,15.0,False,Lodging,Ok,False,False,2022-11-11T20:17:12.686Z,...,4,INCONNU,Hotel,63e546a08f7e73edc5913876,False,False,1,5.0,744861,78.0
3,63f2be19ec2561ca0a25eb4f,63e546a08f7e73edc5913876,53.5,15.8,False,Lodging,Jest super Więcej,True,False,2022-07-14T20:17:15.108Z,...,4,INCONNU,Hotel,63e546a08f7e73edc5913876,False,False,1,5.0,744861,78.0
4,63f2be1dec2561ca0a25eb51,63e546a08f7e73edc5913876,53.5,16.0,False,Shops,Ok,False,False,2022-07-14T20:17:15.313Z,...,4,INCONNU,Sklep dla majsterkowiczów,63e546a08f7e73edc5913876,False,False,1,5.0,744861,78.0


## Sauvegarder données nettoyées

In [25]:
# Créer le répertoire de sortie
from pathlib import Path
output_dir = Path('../data/processed')
output_dir.mkdir(exist_ok=True)

# Sauvegarder les données nettoyées
output_path = output_dir / 'reviews_cleaned.csv'
merged.to_csv(output_path, index=False)

print(f"\n✅ SAVED : {output_path}")
print(f"   Size : {output_path.stat().st_size / 1024**2:.2f} Mo")

# Vérification
df_verify = pd.read_csv(output_path)
print(f"\n✅ VERIFICATION :")
print(f"   Loaded shape : {df_verify.shape}")
print(f"   Columns : {list(df_verify.columns)[:5]}... ({len(df_verify.columns)} total)")


✅ SAVED : ..\data\processed\reviews_cleaned.csv
   Size : 4.22 Mo

✅ VERIFICATION :
   Loaded shape : (13351, 24)
   Columns : ['review_id', 'account_id', 'approximate_localization.lat', 'approximate_localization.lon', 'censored_text']... (24 total)
